In [1]:
from requests_html import HTMLSession
from lxml_html_clean import Cleaner
import pandas as pd

In [ ]:
links = []
with open('..\\data_collected\\product_links.txt', 'r') as f:
    for line in f:
        name, link = line.split(sep= ':')
        links.append((name.strip(), link.strip()))

session = HTMLSession()

fitment_df = pd.DataFrame(columns= ['year', 'make', 'model', 'engine'])
product_df = pd.DataFrame(columns= ['brand', 'part_number', 'type', 'verified_fitments', 'year', 'makes', 'description'])

for idx, link in enumerate(links):
        url = 'https://autofactsdb.com' + link[1]
        r = session.get(url)
        product_attr = [fact.text.split(sep= '\n') for fact in r.html.find('.quick-facts__row')][:6]
        description = [prose.text.split(sep= '\n')[0] for prose in r.html.find('.prose.part-prose')]
    
        table = []
        fitments = r.html.find('tr')
        for i, fitment in enumerate(fitments):
            if i != 0:
                record = fitment.text.split(sep= '\n')
                if len(record) > 4:
                    merger = [record[2] + ' ' + record[3]]
                    record = record[:2] + merger + record[4:]
                table.extend(record)

        product = []
        for p in product_attr:
            product.append(p[1])


        product_df.loc[len(product_df)] = product + description

        i = 0
        while i < len(table):
            fitment_df.loc[len(fitment_df)] = table[i : i + 4]
            i = i + 4

        splitted = fitment_df['engine'].str.split(pat= ' ', n= 2)
        fitment_df['engine'] = splitted.str.get(0) + ' ' + splitted.str.get(1)  



In [ ]:
fitment_df.to_csv('..\\data_collected\\fitment.csv', index= False)
product_df.to_csv('..\\data_collected\\product.csv', index= False)